<a href="https://colab.research.google.com/github/Lainbon/teste/blob/main/Design_Help!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install google-genai

In [ ]:
# Configura a API Key do Google Gemini

import os
from google.colab import userdata
import warnings
import re # Para parsear a saída do agente coordenador
import textwrap

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
# Configura o cliente da SDK do Gemini

from google import genai

client = genai.Client()

MODEL_ID = "gemini-2.0-flash"

In [ ]:
# Exibe a busca
print(f"Busca realizada: {response.candidates[0].grounding_metadata.web_search_queries}")
# Exibe as URLs nas quais ele se baseou
print(f"Páginas utilizadas na resposta: {', '.join([site.web.title for site in response.candidates[0].grounding_metadata.grounding_chunks])}")
print()
display(HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content))

Busca realizada: ['próxima Imersão IA com Google Gemini Alura']
Páginas utilizadas na resposta: starten.tech, alura.com.br



In [ ]:
# Instalar Framework ADK de agentes do Google ################################################
!pip install -q google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.1/232.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.1/217.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.1/334.1 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0/119.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types  # Para criar conteúdos (Content e Part)
from datetime import date
import textwrap # Para formatar melhor a saída de texto
from IPython.display import display, Markdown # Para exibir texto formatado no Colab
import requests # Para fazer requisições HTTP
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# --- 1. Função Auxiliar para Chamar Agentes ADK ---
# Esta função é essencial para executar um agente ADK e obter a resposta final.
def call_agent(agent: Agent, message_text: str) -> str:
    """
    Envia uma mensagem de texto para um agente ADK via Runner e retorna a resposta final.
    """
    # Cria um serviço de sessão em memória para gerenciar o estado da conversa (simples para um único turno/pergunta)
    session_service = InMemorySessionService()
    # Cria uma nova sessão. Use user_id e session_id consistentes se quiser manter histórico real.
    # Para este exemplo simples de um turno, IDs fixos são suficientes.
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    # Cria um Runner para o agente específico
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    # Cria o conteúdo da mensagem de entrada no formato esperado pelo ADK
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    # LINHA DE DEBUG A SER COMENTADA:
    # print(f"[DEBUG ADK] Executando Agente '{agent.name}' com input: '{message_text}'")
    # Executa o agente. O runner orquestra as chamadas ao modelo e uso de ferramentas.
    for event in runner.run(user_id="user1", session_id="session1", new_message=content):
        if event.is_final_response():
            # Concatena as partes de texto da resposta final do agente
            for part in event.content.parts:
                if part.text is not None:
                    final_response += part.text + "\n"
            # Opcional: Você pode querer parar após a primeira resposta final em um sistema de roteamento simples
            # break # Descomente se quiser apenas a primeira resposta final
    # LINHA DE DEBUG A SER COMENTADA:
    # print(f"[DEBUG ADK] Agente '{agent.name}' finalizado. Resposta bruta: '{final_response.strip()}'")
    return final_response.strip() # Retorna a resposta final (removendo espaços extra no início/fim)

In [ ]:
# --- 2. Função auxiliar para exibir texto formatado em Markdown no Colab
def to_markdown(text):
    text = text.replace('•', '  *') # Formata listas do Gemini
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
# Agente 1: Agente Coordenador (Router)
agente_coordenador_instancia = Agent(
    name="agente_coordenador",
    model="gemini-1.5-flash-latest", # Modelo rápido para classificação
    instruction="""
    Você é o recepcionista do bot Design HELP!. Sua tarefa é analisar a pergunta do usuário e identificar qual especialista deve responder.
    Responda SEMPRE e APENAS no formato: [NOME_DO_AGENTE]: [PARTE_RELEVANTE_DA_PERGUNTA]
    NÃO adicione nenhum outro texto antes ou depois do formato [NOME_DO_AGENTE]: [PERGUNTA].

    Nomes de Agentes Válidos:
    - MEDIDAS: Para perguntas sobre tamanhos, dimensões, resoluções, DPI, PPI, unidades de medida (mm, pixels, polegadas, etc.).
    - CORES: Para perguntas sobre códigos de cores (HEX, RGB, CMYK), modelos de cor, teoria da cor, psicologia das cores, harmonias de cores.
    - FORMATOS: Para perguntas sobre tipos de arquivos (JPG, PNG, SVG, PDF, TIFF, GIF) e quando usar cada um, diferenças entre eles.
    - DEFINICOES: Para perguntas pedindo a definição ou explicação de termos e conceitos gerais de design (sangria, grid, layout, hierarquia, contraste, etc.).
    - PADRAO: Para perguntas que não se encaixam nas categorias acima ou são confusas.

    Analise a pergunta do usuário e escolha o agente mais adequado.
    """,
    description="Agente que coordena as perguntas para o agente especialista correto",
    tools=[] # O coordenador NÃO precisa de ferramentas de busca para rotear
)


In [ ]:
# Agente 2: Medidas e Tamanhos
agente_medidas_instancia = Agent(
    name="agente_medidas_tamanhos",
    model="gemini-2.0-flash", # Modelo mais capaz para respostas factuais
    instruction="""
    Você é o Agente de Medidas e Tamanhos do Design HELP!. Responda perguntas sobre dimensões (papel, web, impressão, etc.), resoluções (DPI/PPI) e unidades de medida (mm, pixels, polegadas, pontos, paicas, etc.).
    Use sua base de conhecimento interna (seu treinamento) e, se necessário, a ferramenta de busca do Google para encontrar a informação mais precisa e atualizada, especialmente para padrões web/redes sociais que mudam ou medidas menos comuns.
    Se a pergunta pedir uma conversão (ex: polegadas para mm), realize a conversão.
    Forneça a medida na unidade solicitada (se possível) ou em unidades padrão relevantes para design (mm para impressão, pixels para web).
    Se a pergunta incluir vários termos de medida ou for complexa, desmembre e responda cada parte se puder.
    Se não encontrar a informação exata, diga claramente. Mantenha a resposta clara e direta.
    """,
    description="Fornece medidas, tamanhos e informações sobre resolução.",
    tools=[google_search] # Este agente PODE usar a busca do Google
)


In [ ]:
# Agente 3: Cores
agente_cores_instancia = Agent(
     name="agente_cores",
     model="gemini-2.0-flash", # Modelo capaz para teoria e códigos
     instruction="""
     Você é o Agente de Cores do Design HELP!. Forneça códigos de cores (HEX, RGB, CMYK, HSL, Pantone se conhecido), explique modelos de cor, conceitos básicos de teoria da cor (harmonias, temperatura, etc.) e psicologia das cores.
     Use sua base de conhecimento (treinamento) e a ferramenta de busca do Google para encontrar códigos de cores específicos pelo nome ("código HEX para azul Tiffany"), ou para buscar exemplos visuais de paletas ou estudos sobre psicologia das cores.
     Gere explicações claras e úteis sobre teoria e psicologia das cores.
     Entregue códigos de cores nos formatos solicitados e explicações concisas.
     """,
     description="Fornece códigos de cores e explica teoria e psicologia das cores.",
     tools=[google_search] # Este agente PODE usar a busca
)


In [ ]:
# Agente 4: Formatos
agente_formatos_instancia = Agent(
     name="agente_formatos",
     model="gemini-2.0-flash", # Modelo capaz para explicações detalhadas
     instruction="""
     Você é o Agente de Formatos do Design HELP!. Explique as características de diferentes formatos de imagem e documento (JPG, PNG, SVG, PDF, TIFF, GIF, etc.), seus prós e contras, e quando usar cada um.
     Descreva as características principais de cada formato (compressão, transparência, vetorial vs raster, animação, etc.).
     Use sua base de conhecimento (treinamento) e a ferramenta de busca do Google para encontrar informações muito técnicas sobre um formato específico ou comparar características detalhadas.
     Gere explicações claras e comparativas ("qual a diferença entre JPG e PNG e quando usar cada um?").
     Entregue as informações como descrição do formato, características principais e casos de uso recomendados.
     """,
     description="Explica diferentes formatos de arquivo, suas características e casos de uso.",
     tools=[google_search] # Este agente PODE usar a busca para detalhes técnicos ou comparações
)


In [ ]:
# Agente 5: Definições e Conceitos Gerais
agente_definicoes_instancia = Agent(
     name="agente_definicoes",
     model="gemini-2.0-flash", # Modelo capaz para definições
     instruction="""
     Você é o Agente de Definições e Conceitos Gerais do Design HELP!. Defina e explique termos e conceitos gerais do design gráfico (tipografia, layout, grid, margem, sangria, hierarquia visual, contraste, equilíbrio, etc.).
     Use sua base de conhecimento (treinamento) para fornecer definições claras e concisas.
     Se necessário, use a ferramenta de busca do Google para encontrar definições alternativas, sinônimos ou exemplos de uso em contexto.
     Entregue uma definição clara e útil para o termo ou conceito solicitado.
     """,
     description="Define termos e conceitos gerais de design gráfico.",
     tools=[google_search] # Este agente PODE usar a busca para refinar definições ou encontrar exemplos
)

In [ ]:
# Agente 6: Resposta Padrão (Fallback)
agente_resposta_padrao_instancia = Agent(
    name="agente_resposta_padrao",
    model="gemini-2.0-flash", # Pode ser um modelo mais simples
    instruction="""
    Você é o agente padrão do Design HELP!. Sua função é responder gentilmente quando a pergunta do usuário não foi entendida pelo sistema de roteamento ou quando nenhum agente especialista pôde responder.
    Peça ao usuário para reformular a pergunta ou diga que ainda não possui informação sobre esse tópico específico.
    Mantenha uma postura amigável e útil.
    """,
    description="Lida com perguntas não classificadas ou sem resposta.",
    tools=[] # Não precisa de ferramentas
)

In [ ]:
# --- 4. Mapeamento de Nomes de Agentes para Instâncias ---
# Este dicionário liga o nome que o Agente Coordenador retornará à instância real do Agente ADK.
mapeamento_instancias_agentes = {
    "MEDIDAS": agente_medidas_instancia,
    "CORES": agente_cores_instancia,
    "FORMATOS": agente_formatos_instancia,
    "DEFINICOES": agente_definicoes_instancia,
    "PADRAO": agente_resposta_padrao_instancia
}


In [ ]:
# --- 5. Função para Processar a Pergunta Usando o Coordenador ---
# Esta função chama o agente coordenador via call_agent e parseia a resposta para determinar o próximo agente.
def processar_com_agente_coordenador(pergunta_usuario: str, agente_coordenador_instancia: Agent) -> tuple[str, str]:
    """
    Chama o agente coordenador e parseia a resposta no formato [NOME_AGENTE]: [SUB_PERGUNTA]
    Retorna uma tupla: (nome_do_agente_destino, pergunta_para_esse_agente)
    Em caso de falha no parsing, retorna ("PADRAO", pergunta_original).
    """
    # REMOVENDO DEBUG: print(f"\n[DEBUG COORDENADOR] Recebendo pergunta para rotear: '{pergunta_usuario}'")

    # Chama o Agente Coordenador usando a função auxiliar call_agent
    # O Coordenador usará sua instrução para decidir o routing e retornar o formato estruturado.
    resposta_bruta_coordenador = call_agent(agente_coordenador_instancia, pergunta_usuario)

    # REMOVENDO DEBUG: print(f"[DEBUG COORDENADOR] Resposta bruta do Coordenador: '{resposta_bruta_coordenador}'")

    # --- Parsear a Resposta Estruturada do Coordenador ---
    # Esperamos o formato: [NOME_DO_AGENTE]: [PERGUNTA_PARA_ESSE_AGENTE]
    match = re.match(r"([A-Z_]+):\s*(.*)", resposta_bruta_coordenador)

    if match:
        nome_agente = match.group(1).strip()
        pergunta_para_agente = match.group(2).strip()
        # REMOVENDO DEBUG: print(f"[DEBUG COORDENADOR] Parseado -> Agente: '{nome_agente}', Sub-pergunta: '{pergunta_para_agente}'")
        # Verifica se o nome do agente retornado é válido (opcional, o get do dicionário já trata isso)
        # if nome_agente in mapeamento_instancias_agentes:
        return nome_agente, pergunta_para_agente
        # else:
        #      REMOVENDO DEBUG: print(f"[DEBUG COORDENADOR] Nome de agente '{nome_agente}' retornado pelo Coordenador é inválido.")
        #      return "PADRAO", pergunta_usuario # Roteia para o agente padrão se o nome for inválido
    else:
        # Se o coordenador não seguir o formato esperado, trata como padrão
        # REMOVENDO DEBUG: print(f"[DEBUG COORDENADOR] Formato de resposta do Coordenador inesperado. Tratando como PADRAO.")
        # Passa a pergunta original para o agente padrão
        return "PADRAO", pergunta_usuario

In [ ]:
# --- 6. Função para Processar com o Agente Especialista ---
# Esta função genérica chama a instância do agente especialista correspondente.
def processar_agente_especialista(nome_agente: str, sub_pergunta: str, instancias_agentes: dict[str, Agent]) -> str:
    """
    Encontra a instância do agente especialista pelo nome e a chama com a sub_pergunta.
    Retorna a resposta final do especialista.
    """
    instancia_agente = instancias_agentes.get(nome_agente)

    if instancia_agente:
        # Chama o Agente Especialista usando a função auxiliar call_agent
        # A lógica interna do especialista (usar instrução, ferramentas, modelo) acontece dentro de call_agent/Runner.
        resposta_final = call_agent(instancia_agente, sub_pergunta)
        return resposta_final
    else:
        # Esta situação deve ser rara se o parser do coordenador e o mapeamento estiverem corretos,
        # mas é um fallback final.
        # REMOVENDO DEBUG: print(f"[DEBUG ERRO] Agente '{nome_agente}' não encontrado no mapeamento durante a chamada do especialista.")
        # Chama o agente padrão diretamente como fallback
        return instancias_agentes["PADRAO"].process("Erro interno ao chamar agente especialista.") # Usa o método process direto do agente padrão


In [ ]:
# --- 7. Loop Principal do Chat ---
data_de_hoje = date.today().strftime("%d/%m/%Y") # A data pode ser usada na instrução de algum agente se necessário

print("Bem-vindo ao Design HELP! Pergunte sobre medidas, cores, formatos ou definições de design.")
print("Digite 'sair' para encerrar.")

while True:
    pergunta_do_usuario = input("\nVocê: ")

    if pergunta_do_usuario.lower() == 'sair':
        print("Design HELP: Até mais!")
        break

    if not pergunta_do_usuario.strip():
        print("Design HELP: Por favor, digite sua dúvida.")
        continue # Pula para a próxima iteração se a entrada for vazia

    print("-" * 40) # Separador visual

    try:
        # 1. Processar a pergunta com o Agente Coordenador
        # Ele retorna qual agente chamar e a sub-pergunta para ele
        nome_proximo_agente, sub_pergunta_para_especialista = processar_com_agente_coordenador(
            pergunta_do_usuario,
            agente_coordenador_instancia # Passa a instância do agente coordenador
        )

        # 2. Chamar o Agente Especialista apropriado usando a função de processamento genérica
        resposta_do_bot = processar_agente_especialista(
            nome_proximo_agente,
            sub_pergunta_para_especialista,
            mapeamento_instancias_agentes # Passa o dicionário de mapeamento
        )

        # 3. Exibir a resposta
        print("\n--- Resposta do Design HELP! ---")
        display(to_markdown(resposta_do_bot)) # Exibe formatado como Markdown
        print("-" * 40) # Separador visual

    except Exception as e:
        print(f"[ERRO CRÍTICO] Ocorreu um erro inesperado: {e}")
        print("Design HELP: Desculpe, ocorreu um erro interno. Por favor, tente novamente mais tarde.")
        print("-" * 40) # Separador visual

Bem-vindo ao Design HELP! Pergunte sobre medidas, cores, formatos ou definições de design.
Digite 'sair' para encerrar.

Você: que ano nasceu pelé
----------------------------------------

--- Resposta do Design HELP! ---


> Olá! Desculpe, mas não consegui entender sua pergunta. Você pode tentar reformulá-la ou perguntar sobre outro tópico?

----------------------------------------

Você: sugestões de paleta de cores para uma marca elegente
----------------------------------------

--- Resposta do Design HELP! ---


> Para uma marca elegante, podemos considerar paletas que transmitam sofisticação, confiança e um toque de modernidade. Aqui estão algumas sugestões:
> 
> 1.  **Clássico e Sofisticado:**
> 
>     *   Cores Primárias: Preto e branco
>     *   Cor de Destaque: Dourado ou Prata (tons metálicos)
>     *   Código HEX:
>         *   Preto: #000000
>         *   Branco: #FFFFFF
>         *   Dourado: #FFD700
>         *   Prata: #C0C0C0
>     *   Psicologia: Transmite luxo, minimalismo e atemporalidade.
> 2.  **Elegância Neutra:**
> 
>     *   Cores Primárias: Bege e Cinza
>     *   Cor de Destaque: Rosé ou Bordô
>     *   Código HEX:
>         *   Bege: #F5F5DC
>         *   Cinza: #808080
>         *   Rosé: #F7CAC9
>         *   Bordô: #800000
>     *   Psicologia: Calmante, refinada e com um toque de feminilidade.
> 3.  **Moderno e Chic:**
> 
>     *   Cores Primárias: Azul Marinho e Branco
>     *   Cor de Destaque: Cobre ou Laranja Queimado
>     *   Código HEX:
>         *   Azul Marinho: #000080
>         *   Branco: #FFFFFF
>         *   Cobre: #B87333
>         *   Laranja Queimado: #CC6633
>     *   Psicologia: Confiança, energia sutil e modernidade.
> 4.  **Minimalista:**
> 
>     *   Cores Primárias: Off-White e Cinza Claro
>     *   Cor de Destaque: Verde Oliva ou Terracota
>     *   Código HEX:
>         *   Off-White: #F8F8FF
>         *   Cinza Claro: #D3D3D3
>         *   Verde Oliva: #808000
>         *   Terracota: #E2725B
>     *   Psicologia: Simplicidade, sofisticação discreta e naturalidade.
> 
> Para te ajudar ainda mais, posso buscar exemplos visuais dessas paletas ou informações adicionais sobre a psicologia das cores específicas que você mais gostou. Qual dessas opções te interessa mais?

----------------------------------------

Você: dê exemplos visuais
----------------------------------------

--- Resposta do Design HELP! ---


...

----------------------------------------

Você: exemplos visuais dessas paletas
----------------------------------------

--- Resposta do Design HELP! ---


> Olá! Não entendi muito bem a sua pergunta. Você pode reformulá-la, por favor? Se for sobre paletas de cores, posso te ajudar a encontrar exemplos visuais! 😊

----------------------------------------

Você: Dê exemplos visuais sobre as paletas de cores
----------------------------------------

--- Resposta do Design HELP! ---


> Aqui estão algumas sugestões de pesquisa que podem te ajudar a encontrar exemplos visuais de paletas de cores:
> 
> 
> Para te ajudar a visualizar paletas de cores, posso te dar algumas dicas e exemplos:
> 
> *   **Círculo Cromático:** Uma ferramenta fundamental. Ele mostra as relações entre as cores e ajuda a criar combinações harmoniosas. As combinações mais comuns são:
>     *   **Monocromáticas:** Usam variações de uma única cor.
>     *   **Análogas:** Cores próximas umas das outras no círculo cromático.
>     *   **Complementares:** Cores opostas no círculo, criando contraste.
>     *   **Tríades:** Três cores equidistantes no círculo, resultando em combinações vibrantes.
> 
> *   **Paletas por Emoção:**
>     *   **Cores Quentes:** Vermelhos, amarelos e laranjas transmitem energia, paixão e alegria.
>     *   **Cores Frias:** Azuis, verdes e roxos evocam calma, serenidade e sofisticação.
> 
> *   **Exemplos Visuais:**
>     *   **Natureza:** Inspire-se em paisagens, flores e elementos naturais para criar paletas orgânicas e equilibradas.
>     *   **Design:** Explore sites como Pinterest, Dribbble e Behance para encontrar paletas de cores usadas em design gráfico, web design e decoração.
> 
> Se você tiver alguma preferência de cor ou tema em mente, me diga! Posso te ajudar a encontrar exemplos mais específicos.

----------------------------------------

Você: sair
Design HELP: Até mais!
